In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/drsinglabhawnasingla/multi-modal-carnatic-music-dataset/Chakkani Raja.npz
/kaggle/input/datasets/drsinglabhawnasingla/multi-modal-carnatic-music-dataset/Varashiki Vahana.npz
/kaggle/input/datasets/drsinglabhawnasingla/multi-modal-carnatic-music-dataset/Thappi Bratikipova.npz
/kaggle/input/datasets/drsinglabhawnasingla/multi-modal-carnatic-music-dataset/Manasaramathi.npz
/kaggle/input/datasets/drsinglabhawnasingla/multi-modal-carnatic-music-dataset/Pavamana Suthudu Mangalam.npz
/kaggle/input/datasets/drsinglabhawnasingla/multi-modal-carnatic-music-dataset/Shobillu Saptasvara.npz
/kaggle/input/datasets/drsinglabhawnasingla/multi-modal-carnatic-music-dataset/Avani Sutha.npz
/kaggle/input/datasets/drsinglabhawnasingla/multi-modal-carnatic-music-dataset/Bhuvini Dasudane.npz
/kaggle/input/datasets/drsinglabhawnasingla/multi-modal-carnatic-music-dataset/Apparama Bhakti.npz
/kaggle/input/datasets/drsinglabhawnasingla/multi-modal-carnatic-music-dataset/Shloka.npz
/kaggle/

In [2]:
# ==========================================================
# Cell 1 : Imports
# ==========================================================

import os
import glob
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

print("PyTorch:", torch.__version__)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device :", DEVICE)

PyTorch: 2.10.0+cpu
Device : cpu


In [3]:
# ==========================================================
# Cell 2 : Configuration
# ==========================================================

DATASET_DIR = "/kaggle/input/datasets/drsinglabhawnasingla/multi-modal-carnatic-music-dataset"

EMBED_DIM = 128

NUM_CLASSES = 12

WINDOW_SIZE = 512

STRIDE = 256

BATCH_SIZE = 32

In [4]:
# ==========================================================
# Cell 3 : Locate Dataset
# ==========================================================

files = sorted(
    glob.glob(
        os.path.join(DATASET_DIR, "**", "*.npz"),
        recursive=True
    )
)

print("="*60)
print("Songs :", len(files))
print("="*60)

print(files[:5])

Songs : 114
['/kaggle/input/datasets/drsinglabhawnasingla/multi-modal-carnatic-music-dataset/Aaniraimekkani.npz', '/kaggle/input/datasets/drsinglabhawnasingla/multi-modal-carnatic-music-dataset/Amba Nilambari.npz', '/kaggle/input/datasets/drsinglabhawnasingla/multi-modal-carnatic-music-dataset/Apparama Bhakti.npz', '/kaggle/input/datasets/drsinglabhawnasingla/multi-modal-carnatic-music-dataset/Ardhanareeshwaram.npz', '/kaggle/input/datasets/drsinglabhawnasingla/multi-modal-carnatic-music-dataset/Ardhanarishwaram.npz']


In [5]:
# ==========================================================
# Cell 4 : Verify Dataset
# ==========================================================

sample = np.load(files[0], allow_pickle=True)

print(sample.files)

print()

print("Feature Shape :", sample["X"].shape)

print("Label Shape   :", sample["y"].shape)

print("Tonic         :", sample["tonic"])

print("Song          :", sample["song"])

['X', 'y', 'tonic', 'song']

Feature Shape : (28710, 66)
Label Shape   : (28710,)
Tonic         : 191.52112
Song          : Aaniraimekkani


In [6]:
# ==========================================================
# Cell 5 : Train Validation Split
# ==========================================================

train_files, val_files = train_test_split(

    files,

    test_size=0.2,

    random_state=42,

    shuffle=True

)

print("Train :", len(train_files))

print("Validation :", len(val_files))

Train : 91
Validation : 23


In [7]:
# ==========================================================
# Cell 6 : Window Dataset
# ==========================================================

class SwaraDataset(Dataset):

    def __init__(self, files,
                 window_size=512,
                 stride=256):

        self.samples = []

        for file in files:

            data = np.load(file, allow_pickle=True)

            X = data["X"]

            y = data["y"]

            T = len(y)

            for start in range(0,
                               T-window_size,
                               stride):

                end = start + window_size

                self.samples.append(

                    (

                        X[start:end],

                        y[start:end]

                    )

                )

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        X, y = self.samples[idx]

        return (

            torch.tensor(X, dtype=torch.float32),

            torch.tensor(y, dtype=torch.long)

        )

In [8]:
# ==========================================================
# Cell 7 : Build Dataset
# ==========================================================

train_dataset = SwaraDataset(

    train_files,

    WINDOW_SIZE,

    STRIDE

)

val_dataset = SwaraDataset(

    val_files,

    WINDOW_SIZE,

    STRIDE

)

print()

print("Training Windows :", len(train_dataset))

print("Validation Windows :", len(val_dataset))


Training Windows : 27515
Validation Windows : 6224


In [10]:
# ==========================================================
# Cell 8 : DataLoader
# ==========================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=2

)

val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=2

)

In [11]:
# ==========================================================
# Cell 9 : Test Batch
# ==========================================================

X, y = next(iter(train_loader))

print("="*60)

print("Input :", X.shape)

print("Labels:", y.shape)

Input : torch.Size([32, 512, 66])
Labels: torch.Size([32, 512])


In [12]:
# ==========================================================
# Cell 10 : Small Subset
# ==========================================================

train_subset = train_files[:20]

val_subset = val_files[:5]

print("Training Songs :", len(train_subset))

print("Validation Songs :", len(val_subset))

Training Songs : 20
Validation Songs : 5


In [13]:
# ==========================================================
# Cell 11 : Dataset
# ==========================================================

train_dataset = SwaraDataset(

    train_subset,

    window_size=512,

    stride=256

)

val_dataset = SwaraDataset(

    val_subset,

    window_size=512,

    stride=256

)

print()

print("Training Windows :", len(train_dataset))

print("Validation Windows :", len(val_dataset))


Training Windows : 7836
Validation Windows : 1036


In [14]:
# ==========================================================
# Cell 12 : DataLoader
# ==========================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=32,

    shuffle=True

)

val_loader = DataLoader(

    val_dataset,

    batch_size=32,

    shuffle=False

)

print("Done")

Done


In [15]:
# ==========================================================
# Cell 13 : Verify Batch
# ==========================================================

X, y = next(iter(train_loader))

print("Input Shape :", X.shape)

print("Label Shape :", y.shape)

print()

print("Input dtype :", X.dtype)

print("Label dtype :", y.dtype)

Input Shape : torch.Size([32, 512, 66])
Label Shape : torch.Size([32, 512])

Input dtype : torch.float32
Label dtype : torch.int64


In [16]:
# ==========================================================
# Cell 14 : Shared Encoder
# ==========================================================

class InputEncoder(nn.Module):

    def __init__(self,
                 input_dim=66,
                 embed_dim=128,
                 dropout=0.2):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(input_dim, embed_dim),

            nn.LayerNorm(embed_dim),

            nn.GELU(),

            nn.Dropout(dropout)

        )

    def forward(self, x):

        return self.encoder(x)

In [17]:
# ==========================================================
# Cell 15 : Test Encoder
# ==========================================================

encoder = InputEncoder()

encoder = encoder.to(DEVICE)

X = X.to(DEVICE)

with torch.no_grad():

    out = encoder(X)

print()

print("Input :", X.shape)

print("Output:", out.shape)


Input : torch.Size([32, 512, 66])
Output: torch.Size([32, 512, 128])


In [18]:
# ==========================================================
# Cell 16 : Positional Encoding
# ==========================================================

import math

class PositionalEncoding(nn.Module):

    def __init__(self,
                 d_model=128,
                 max_len=5000):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(
            0,
            max_len,
            dtype=torch.float
        ).unsqueeze(1)

        div_term = torch.exp(

            torch.arange(
                0,
                d_model,
                2
            ).float()

            * (-math.log(10000.0) / d_model)

        )

        pe[:, 0::2] = torch.sin(position * div_term)

        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):

        return x + self.pe[:, :x.size(1)]

In [19]:
# ==========================================================
# Cell 17 : Test Positional Encoding
# ==========================================================

pos_encoder = PositionalEncoding()

pos_encoder = pos_encoder.to(DEVICE)

with torch.no_grad():

    out = pos_encoder(out)

print(out.shape)

torch.Size([32, 512, 128])


In [20]:
# ==========================================================
# Cell 18 : Shared Feature Encoder
# ==========================================================

class SharedEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.input = InputEncoder()

        self.position = PositionalEncoding()

    def forward(self, x):

        x = self.input(x)

        x = self.position(x)

        return x

In [21]:
# ==========================================================
# Cell 19 : Test Shared Encoder
# ==========================================================

model = SharedEncoder().to(DEVICE)

with torch.no_grad():

    output = model(X)

print("="*50)

print("Input Shape :", X.shape)

print("Output Shape:", output.shape)

Input Shape : torch.Size([32, 512, 66])
Output Shape: torch.Size([32, 512, 128])


In [22]:
# ==========================================================
# Cell 20 : Save Encoder
# ==========================================================

os.makedirs("models", exist_ok=True)

torch.save(

    model.state_dict(),

    "models/shared_encoder.pth"

)

print("Shared Encoder Saved Successfully.")

Shared Encoder Saved Successfully.
